# 1. Objective

The goal of this notebook is to design the extraction pipeline that converts one raw Slay the Spire run into the canonical decision dataset defined in Notebook 01.

# 2. Run Inspection

Loading an Ironclad run from the raw dataset.

In [29]:
# Reload raw run data (each notebook has its own kernel/session)
import gzip
import json

with gzip.open("../data/raw/2020-09-28-04-45_965.json.gz", "rt", encoding="utf-8") as f:
    runs = json.load(f)

# Filter to Ironclad runs, per DD-003
ironclad_runs = [r['event'] for r in runs if r['event'].get('character_chosen') == 'IRONCLAD']

# Filter to won Ironclad runs, 
won_ironclad_runs = [r for r in ironclad_runs if r.get('victory') == True]

# Inspect how many Ironclad runs exist in the raw data
print(f"Number of Ironclad runs: {len(ironclad_runs)}")
print(f"Number of won Ironclad runs: {len(won_ironclad_runs)}")

Number of Ironclad runs: 342
Number of won Ironclad runs: 31


Inspection of the main characteristics of a run as specified in the dataset schema in Notebook 1.

In [30]:
lost_run_example = ironclad_runs[0]

print(lost_run_example.keys())
print("Card Choices:", lost_run_example['card_choices'])
print("Master Deck:", lost_run_example['master_deck'])
print("Relics:", lost_run_example['relics'])
print("Current HP per Floor:", lost_run_example['current_hp_per_floor'])
print(type(lost_run_example['current_hp_per_floor']))
print("Max HP per Floor:", lost_run_example['max_hp_per_floor'])
print("Gold per Floor:", lost_run_example['gold_per_floor'])
print("Floor Reached:", lost_run_example['floor_reached'])


dict_keys(['gold_per_floor', 'floor_reached', 'playtime', 'items_purged', 'score', 'play_id', 'local_time', 'is_ascension_mode', 'campfire_choices', 'neow_cost', 'seed_source_timestamp', 'circlet_count', 'master_deck', 'relics', 'potions_floor_usage', 'damage_taken', 'seed_played', 'potions_obtained', 'is_trial', 'path_per_floor', 'character_chosen', 'items_purchased', 'campfire_rested', 'item_purchase_floors', 'current_hp_per_floor', 'gold', 'neow_bonus', 'is_prod', 'is_daily', 'chose_seed', 'campfire_upgraded', 'win_rate', 'timestamp', 'path_taken', 'build_version', 'purchased_purges', 'victory', 'max_hp_per_floor', 'card_choices', 'player_experience', 'relics_obtained', 'event_choices', 'is_beta', 'boss_relics', 'items_purged_floors', 'is_endless', 'potions_floor_spawned', 'killed_by', 'ascension_level'])
Card Choices: [{'not_picked': ['Clothesline', 'Rage', 'Infernal Blade'], 'picked': 'SKIP', 'floor': 1}, {'not_picked': ['Thunderclap', 'Pummel', 'Searing Blow'], 'picked': 'SKIP', 

The most important fields are the following:
- 'card_choices': a disctionary which contains all the offered cards and the chosed card or 'SKIP'
- 'master_deck': a list which contains the whole deck at the end of the run 
- 'relics': a list of all the relics accumulated by the end of the run
- 'current_hp_per_floor': a list containing the hp of the player on every floor
- 'max_hp_per_floor': a list containing the max hp of the player on every floor
- 'gold_per_floor': a list containing the current gold of the player on every floor
- 'floor_reached': The floor that the player finished his run

The inspection provided an overview of the available information. Before designing the extraction algorithm, the reliability and consistency of these fields must be verified.

# 3. Dataset Validation

## Finding F-001 — 'current_hp_per_floor' is not a reliable floor index

### Question: Does current_hp_per_floor correspond one-to-one with floors?

Noticed an inconsistency between the number of current_hp_per_floor and floor_reached, which is worth investigating. A likely explanation is that it is a special seed or some other peculiarity.

In [31]:
#Inspect the keys of a lost Ironclad run

print("Is Trial:", lost_run_example['is_trial'])
print("Is Daily:", lost_run_example['is_daily'])
print("Chose Seed:", lost_run_example['chose_seed'])
print("Path Taken:", lost_run_example['path_taken'])
print("Killed By:", lost_run_example['killed_by'])

print("Number of times Hp per floor is stored in a lost run:", len(lost_run_example['current_hp_per_floor']))
print("Floor Reached:", lost_run_example['floor_reached'])


Is Trial: False
Is Daily: False
Chose Seed: False
Path Taken: ['M', 'M', '?', '$', '?']
Killed By: 2 Louse
Number of times Hp per floor is stored in a lost run: 6
Floor Reached: 5


In [32]:
#Inspect the keys of a won Ironclad run

won_run_example = won_ironclad_runs[0]

print(won_run_example.keys())
print("Card Choices:",won_run_example['card_choices'])
print("Master Deck:",won_run_example['master_deck'])
print("Relics:",won_run_example['relics'])
print("Current HP per Floor:",won_run_example['current_hp_per_floor'])
print("Max HP per Floor:",won_run_example['max_hp_per_floor'])
print("Gold per Floor:",won_run_example['gold_per_floor'])
print("Floor Reached:",won_run_example['floor_reached'])   
print("Is Trial:",won_run_example['is_trial'])
print("Is Daily:",won_run_example['is_daily'])
print("Chose Seed:",won_run_example['chose_seed'])
print("Path Taken:",won_run_example['path_taken'])


print("Number of times Hp per floor is stored in a won run:", len(won_run_example['current_hp_per_floor']))
print("Floor Reached:", won_run_example['floor_reached'])


dict_keys(['gold_per_floor', 'floor_reached', 'playtime', 'items_purged', 'score', 'play_id', 'local_time', 'is_ascension_mode', 'campfire_choices', 'neow_cost', 'seed_source_timestamp', 'circlet_count', 'master_deck', 'special_seed', 'relics', 'potions_floor_usage', 'damage_taken', 'seed_played', 'potions_obtained', 'is_trial', 'path_per_floor', 'character_chosen', 'items_purchased', 'campfire_rested', 'item_purchase_floors', 'current_hp_per_floor', 'gold', 'neow_bonus', 'is_prod', 'is_daily', 'chose_seed', 'campfire_upgraded', 'win_rate', 'timestamp', 'path_taken', 'build_version', 'purchased_purges', 'victory', 'max_hp_per_floor', 'card_choices', 'player_experience', 'relics_obtained', 'event_choices', 'is_beta', 'boss_relics', 'items_purged_floors', 'is_endless', 'potions_floor_spawned', 'ascension_level'])
Card Choices: [{'not_picked': ['Heavy Blade', 'Sword Boomerang'], 'picked': 'Flex', 'floor': 1.0}, {'not_picked': ['Pommel Strike', 'Shrug It Off'], 'picked': 'Whirlwind', 'floo

**Result:** there is an inconcistency between the lost run and the won run. In the lost run :<br>
len(current_hp_per_floor) = floor_reached + 1 <br>
In the won run:<br>
len(current_hp_per_floor) = floor_reached - 1 <br>
That's an actual 2 entry swing between cases which means there isn't a single simple offset rule (like "always add 1 for starting HP").

In [33]:
#Compare the lengths of current_hp_per_floor and path_per_floor for lost and won runs

print("Length of current_hp_per_floor for lost runs:", len(lost_run_example['current_hp_per_floor']))
print("Floor Reached for lost runs:", lost_run_example['floor_reached'])
print("Length of path_per_floor for lost runs:", len(lost_run_example['path_per_floor']))

print("Length of current_hp_per_floor for won runs:", len(won_run_example['current_hp_per_floor']))
print("Floor Reached for won runs:", won_run_example['floor_reached'])
print("Length of path_per_floor for won runs:", len(won_run_example['path_per_floor']))

Length of current_hp_per_floor for lost runs: 6
Floor Reached for lost runs: 5
Length of path_per_floor for lost runs: 5
Length of current_hp_per_floor for won runs: 50
Floor Reached for won runs: 51
Length of path_per_floor for won runs: 51


We see that 'path_per_floor' matches 'floor_reached' in both cases (5 and 5 for the death run, 51 and 51 for the victory run). So 'path_per_floor' is the stable, trustworthy reference: one entry per floor actually visited, consistent regardless of how the run ended.

### Decision : `current_hp_per_floor` indexing is inconsistent
Comparing against `path_per_floor` (which reliably matches `floor_reached` in both examples tested) revealed that `current_hp_per_floor`'s length doesn't align consistently:
- Death run (floor_reached=5): 6 entries (one extra).
- Victory run (floor_reached=51): 50 entries (one short).

`path_per_floor` will be used as the reliable floor-count reference. HP alignment for the final floor of a run is treated as unreliable and will be handled explicitly in Section 4.

## Finding F-002: 'killed_by' key value is missing from the won run.

### Question: Which keys are inconsistently present across runs?

In [34]:
# Compare the number of keys in lost and won runs, and check for the presence of 'killed_by' key

print("Number of keys in lost runs:", len(lost_run_example.keys()))
print("killed_by in lost runs:", 'killed_by' in lost_run_example)

print("Number of keys in won runs:", len(won_run_example.keys()))
print("killed_by in won runs:", 'killed_by' in won_run_example)

Number of keys in lost runs: 49
killed_by in lost runs: True
Number of keys in won runs: 49
killed_by in won runs: False


In [35]:
#Find the keys that are present in one run but not the other

death_keys = set(lost_run_example.keys())
victory_keys = set(won_run_example.keys())

print("In death run but not victory run:", death_keys - victory_keys)
print("In victory run but not death run:", victory_keys - death_keys)

In death run but not victory run: {'killed_by'}
In victory run but not death run: {'special_seed'}


In [36]:
# Does the presence of a special seed correlate with victory? Let's check.

has_special_seed = [('special_seed' in r['event']) for r in runs]
print(sum(has_special_seed), "out of", len(runs))

# cross-check against victory, to see if there's a real relationship or not
import collections
victory_by_has_field = collections.Counter(
    (r['event'].get('victory'), 'special_seed' in r['event']) for r in runs
)
print(victory_by_has_field)

323 out of 965
Counter({(False, False): 600, (False, True): 279, (True, True): 44, (True, False): 42})


**Result:** `special_seed` is missing entirely (not just False) in ~33% of runs (323/965). No relationship found with `victory`, likely an unrelated data logging inconsistency.

In [37]:
#Counting the number of times each key appears in the runs, to see if there are any keys that are not present in all runs

import collections

key_counts = collections.Counter()
for r in runs:
    key_counts.update(r['event'].keys())

total_runs = len(runs)
for key, count in sorted(key_counts.items(), key=lambda x: x[1]):
    if count < total_runs:
        print(f"{key}: present in {count}/{total_runs} ({count/total_runs:.1%})")

special_seed: present in 323/965 (33.5%)
killed_by: present in 753/965 (78.0%)


### Decision: Filtering logic must use `.get('special_seed', False)` rather than direct indexing to avoid KeyErrors.

## Finding F-003: There exist runs that were abandoned

### Question: Are there runs that resulted in a loss that didn't finish due to a 'killed_by'?


In [38]:
#Checking how many runs are victories

victories = sum(1 for r in runs if r['event'].get('victory'))
print(victories, "victories out of", len(runs))

86 victories out of 965


Our hypothesis of 'killed_by' missing only from the won runs is not correct since the numbers 86 and 965-753=212 do not allign. This is an actual discrepancy and not just noise. We use the same technique as with the 'special_seed' cross-tab, to see from which runs the key value 'killed_by' is missing.

In [39]:
#Counting how many runs are killed by something, and cross-checking against victory

import collections
killed_by_crosstab = collections.Counter(
    (r['event'].get('victory'), 'killed_by' in r['event']) for r in runs
)
print(killed_by_crosstab)

Counter({(False, True): 753, (False, False): 126, (True, False): 86})


Notice there's no (True, True) combination at all, so no victory run has 'killed_by'. Since 86 'killed_by' are missing from the Victories the question is where the rest 126 are missnig from. A likely candidate would be: runs abandoned/quit before dying or winning.

In [40]:
#Counting how many runs are abandoned (i.e., not victorious and not killed by anything)

abandoned_candidates = [r['event'] for r in runs 
                         if r['event'].get('victory') == False 
                         and 'killed_by' not in r['event']]

print(len(abandoned_candidates))

#Check the first abandoned candidate for floor_reached, path_per_floor, and card_choices
print(abandoned_candidates[0].get('floor_reached'))
print(abandoned_candidates[0].get('path_per_floor'))
print(abandoned_candidates[0].get('card_choices'))

126
0
[]
[]


In [41]:
#Check the floor_reached values for all abandoned candidates

floor_reached_values = [r.get('floor_reached') for r in abandoned_candidates]
print(set(floor_reached_values))

{0, 2, 3, 4, 5, 6, 7, 8, 11, 13, 14, 15, 17, 19, 20, 21, 22, 30}


**Result**: `floor_reached` values for the 126 non-death, non-victory runs range from 0 to 30, confirming these are not all trivial floor-0 quits. Many contain real `card_choices` made under normal play conditions before the player stopped for unrelated reasons.

### Decision: flag missing keys via `is_abandoned` (`victory == False AND 'killed_by' not in run`). Retain them in the canonical and behavioral (Phase 3) dataset, since the decisions themselves are valid signal. Exclude them from any future model using `victory` as a label or sample weight, since an abandoned run's outcome reflects the player stopping, not decision quality. Tracked as DD-011 in Notebook 01.

## Finding F-004 Can the deck be reconstructed exactly at each card choice?

### Question: How often do card removals (purges) occur in Ironclad runs, and could they interfere with reconstructing the deck from `card_choices` alone?

In [42]:
# Counting the number of purges in Ironclad runs, and how many runs have at least one purge

import collections

purge_counts = collections.Counter()
for r in ironclad_runs:
    purged = r.get('items_purged', [])
    purge_counts[len(purged)] += 1

print(purge_counts)

runs_with_purges = sum(1 for r in ironclad_runs if len(r.get('items_purged', [])) > 0)
print(f"{runs_with_purges} out of {len(ironclad_runs)} runs have at least one purge")

Counter({0: 194, 1: 79, 2: 44, 3: 13, 4: 10, 5: 2})
148 out of 342 runs have at least one purge


In [43]:
# Picking a run with at least one purge to inspect the items_purged and items_purged_floors fields

purged_example = next(r for r in ironclad_runs if len(r.get('items_purged', [])) > 0)
print(purged_example.get('items_purged'))
print(purged_example.get('items_purged_floors'))

['Strike_R']
[4]


In [44]:
# Checking if the floor the purge happened on is considtent with a gamestate event, by comparing the items_purged_floors with the path_per_floor

purge_floors = purged_example.get('items_purged_floors')
items_purged = purged_example.get('items_purged')
path = purged_example.get('path_per_floor')

print("Items purged:", items_purged)
print("Purge floors:", purge_floors)

for item, floor in zip(items_purged, purge_floors):
    # path_per_floor is 0-indexed by list position; floor numbers in the 
    # data start at 1, so floor N corresponds to path[N-1]
    room_type = path[int(floor) - 1] if 0 <= int(floor) - 1 < len(path) else "OUT OF RANGE"
    print(f"'{item}' purged on floor {floor} -> room type: {room_type}")

print("Path per floor:", path)


Items purged: ['Strike_R']
Purge floors: [4]
'Strike_R' purged on floor 4 -> room type: $
Path per floor: ['M', 'M', '?', '$', 'M']


In [45]:
import collections

# Count how many purges occur in each room type across all Ironclad runs
room_type_counts = collections.Counter()
# Count how many out of range floor references we encounter
out_of_range_count = 0
# Count how many runs have mismatched lengths of items_purged and items_purged_floors
mismatched_lengths = 0

for r in ironclad_runs:
    items_purged = r.get('items_purged', [])
    purge_floors = r.get('items_purged_floors', [])
    path = r.get('path_per_floor', [])
    
    if len(items_purged) != len(purge_floors):
        mismatched_lengths += 1
        continue
    
    for floor in purge_floors:
        idx = int(floor) - 1
        if 0 <= idx < len(path):
            room_type_counts[path[idx]] += 1
        else:
            out_of_range_count += 1

print("Room types where purges occurred:", room_type_counts)
print("Out-of-range floor references:", out_of_range_count)
print("Runs with mismatched items_purged / items_purged_floors lengths:", mismatched_lengths)

Room types where purges occurred: Counter({'$': 256})
Out-of-range floor references: 0
Runs with mismatched items_purged / items_purged_floors lengths: 0


**Result:** Card purges are common (148/342 Ironclad runs, ~43%) and 
therefore cannot be ignored in deck reconstruction. Cross-checking 
`items_purged_floors` against `path_per_floor` across all runs confirms 
perfect alignment: all 256 purges occur on shop floors ('$'), with zero 
out-of-range floor references and zero length mismatches between 
`items_purged` and `items_purged_floors`.

**Decision:** Deck reconstruction will incorporate purges as floor-indexed removal events, using the same floor-alignment approach as `card_choices` 
additions). `items_purged`/`items_purged_floors` are confirmed reliable for this purpose.

### Question: How often do transformations (either by `neow_bonus`, or mid-run) occur in Ironclad runs, and could they interfere with reconstructing the deck from `card_choices` alone?

In [46]:
STARTING_DECK_SIZE = 10  # 5 Strike, 4 Defend, 1 Bash, pending confirmation on if this matches the data

# Countuing to check for mismatches between expected and actual master deck sizes in Ironclad runs
mismatches = []

for r in ironclad_runs:
    card_choices = r.get('card_choices', [])
    picks = sum(1 for c in card_choices if c.get('picked') != 'SKIP')
    
    purges = len(r.get('items_purged', []))
    
    expected_size = STARTING_DECK_SIZE + picks - purges
    actual_size = len(r.get('master_deck', []))
    
    if expected_size != actual_size:
        mismatches.append({
            'expected': expected_size,
            'actual': actual_size,
            'diff': actual_size - expected_size,
            'picks': picks,
            'purges': purges,
        })

print(f"{len(mismatches)} out of {len(ironclad_runs)} runs have a mismatch")

import collections
diff_distribution = collections.Counter(m['diff'] for m in mismatches)
print("Distribution of size differences:", diff_distribution)

278 out of 342 runs have a mismatch
Distribution of size differences: Counter({1: 82, 2: 45, 3: 40, 5: 24, -1: 22, 4: 16, 7: 13, 6: 12, -3: 5, -2: 4, 8: 4, -4: 2, 12: 2, 9: 2, -11: 1, -12: 1, -5: 1, -7: 1, -6: 1})


In [47]:
#Checking five runs with zero picks and zero purges, which should have a master deck size equal to the starting deck size

zero_pick_zero_purge_runs = [
    r for r in ironclad_runs
    if sum(1 for c in r.get('card_choices', []) if c.get('picked') != 'SKIP') == 0
    and len(r.get('items_purged', [])) == 0
]

print(len(zero_pick_zero_purge_runs), "runs with zero picks and zero purges")

for r in zero_pick_zero_purge_runs[:5]:
    print(len(r.get('master_deck', [])), r.get('master_deck'))

44 runs with zero picks and zero purges
8 ['Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Bash']
12 ['AscendersBane', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Bash', 'Fiend Fire']
10 ['Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Bash']
10 ['Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Bash', 'Flame Barrier', 'Perfected Strike']
11 ['AscendersBane', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Bash']


In [48]:
# Associate the neow_bonus with the master_deck for these runs to see if there's any correlation

for r in zero_pick_zero_purge_runs[:5]:
    print(r.get('neow_bonus'), '->', r.get('master_deck'))

REMOVE_TWO -> ['Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Bash']
ONE_RANDOM_RARE_CARD -> ['AscendersBane', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Bash', 'Fiend Fire']
BOSS_RELIC -> ['Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Bash']
TRANSFORM_TWO_CARDS -> ['Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Bash', 'Flame Barrier', 'Perfected Strike']
THREE_ENEMY_KILL -> ['AscendersBane', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Bash']


In [49]:
# Cross checking the neow_bonus with the master_deck size for these runs to see if there's any mismatch between the expected and actual master deck sizes

neow_deck_size_effects = {
    'REMOVE_TWO': -2,
    'ONE_RANDOM_RARE_CARD': +1,
    'BOSS_RELIC': 0,
    'TRANSFORM_TWO_CARDS': 0,  # same count, different cards
    'THREE_ENEMY_KILL': 0,
}

mismatches = []
for r in zero_pick_zero_purge_runs:
    bonus = r.get('neow_bonus')
    actual_size = len(r.get('master_deck', []))
    #account for AscendersBane, which adds a card to the starting deck
    ascenders_bane = 1 if 'AscendersBane' in r.get('master_deck', []) else 0
    
    expected_size = STARTING_DECK_SIZE + neow_deck_size_effects.get(bonus, 0) + ascenders_bane
    
    if expected_size != actual_size:
        mismatches.append((bonus, expected_size, actual_size, r.get('master_deck')))

print(f"{len(mismatches)} out of {len(zero_pick_zero_purge_runs)} mismatched")
for m in mismatches:
    print(m)

2 out of 44 mismatched
('THREE_ENEMY_KILL', 11, 10, ['AscendersBane', 'Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R+1', 'Defend_R+1', 'Defend_R', 'Defend_R', 'Bash'])
('THREE_ENEMY_KILL', 10, 9, ['Strike_R', 'Strike_R', 'Strike_R', 'Strike_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Defend_R', 'Regret'])


In [50]:
# Review the event_choices for the mismatched runs to see if there's any correlation with the neow_bonus and master_deck size

for bonus, expected, actual, deck in mismatches:
    matching_run = next(r for r in zero_pick_zero_purge_runs 
                         if r.get('neow_bonus') == bonus and len(r.get('master_deck', [])) == actual)
    print(f"--- Run: neow_bonus={bonus}, expected={expected}, actual={actual} ---")
    for choice in matching_run.get('event_choices'):
        print(choice)
    print()  # blank line between runs

--- Run: neow_bonus=THREE_ENEMY_KILL, expected=11, actual=10 ---
{'cards_removed': ['Strike_R'], 'damage_healed': 0, 'gold_gain': 0, 'player_choice': 'Card Removal', 'damage_taken': 0, 'max_hp_gain': 0, 'max_hp_loss': 0, 'event_name': 'The Cleric', 'floor': 2, 'gold_loss': 75}
{'damage_healed': 0, 'gold_gain': 0, 'player_choice': 'Entered Light', 'damage_taken': 23, 'max_hp_gain': 0, 'max_hp_loss': 0, 'event_name': 'Shining Light', 'floor': 3, 'gold_loss': 0, 'cards_upgraded': ['Defend_R', 'Defend_R']}
{'damage_healed': 0, 'gold_gain': 50, 'player_choice': 'Pray', 'damage_taken': 0, 'max_hp_gain': 0, 'max_hp_loss': 0, 'event_name': 'Golden Shrine', 'floor': 5, 'gold_loss': 0}

--- Run: neow_bonus=THREE_ENEMY_KILL, expected=10, actual=9 ---
{'damage_healed': 0, 'gold_gain': 0, 'player_choice': 'Take Damage', 'damage_taken': 20, 'relics_obtained': ['Golden Idol'], 'max_hp_gain': 0, 'max_hp_loss': 0, 'event_name': 'Golden Idol', 'floor': 2, 'gold_loss': 0}
{'cards_removed': ['Bash'], 'dam

**Result:** 5 zero-pick, zero-purge runs were checked against the `neow_bonus`,adjusted with the starting deck hypothesis. All showed deck 
size/composition fully explained by `neow_bonus` (`REMOVE_TWO`, `ONE_RANDOM_RARE_CARD`, `TRANSFORM_TWO_CARDS` all directly account for deviations from the 10-card standard start) plus `AscendersBane`, which is independently explained by ascension level 10+ (already in scope per inclusion criteria), not by Neow.

A follow-up check across all 44 zero-pick, zero-purge runs surfaced 2 further mismatches. Both were fully traced and explained by `event_choices`, not by a gap in the Neow hypothesis:

1. **Run A** (`neow_bonus=THREE_ENEMY_KILL`, expected 11, actual 10): `'The Cleric'` (floor 2) removes `Strike_R`. Explains the deficit exactly.

2. **Run B** (`neow_bonus=THREE_ENEMY_KILL`, expected 10, actual 9): `'Living Wall'` (floor 3) removes `Bash`, `'Golden Wing'` (floor 11) removes `Strike_R`, and `'Big Fish'` (floor 12) adds `Regret` (a curse card). Net effect: −1 −1 +1 = −1, giving exactly 9 cards: 4× Strike_R, 4× Defend_R, 1× Regret — matching `master_deck` exactly, card for card, not just by count.

**Decision:** The starting deck must be Neow-adjusted (not assumed fixed at 10 cards) when reconstructing deck state. This is confirmed by the 5-run sample and further supported by the 2 traced mismatches above, both fully explained by `event_choices`, not by any gap in the Neow hypothesis. Mid-run modifications (transformations, removals, additions) are handled separately via `event_choices`, validated later in this notebook.

### Question: Do `event_choices` ever add/remove/transform cards outside `card_choices`?

In [51]:
# Checking events from the event_choices field in Ironclad runs, to see what kind of data it contains

event_run_sample = next(r for r in won_ironclad_runs if r.get('event_choices'))
print(type(event_run_sample['event_choices']))
print(len(event_run_sample['event_choices']))
for event in event_run_sample['event_choices'][:5]:
    print(event)

<class 'list'>
5
{'damage_healed': 0.0, 'max_hp_gain': 0.0, 'max_hp_loss': 0.0, 'gold_gain': 0.0, 'event_name': 'WeMeetAgain', 'player_choice': 'Gave Potion', 'floor': 4.0, 'gold_loss': 0.0, 'damage_taken': 0.0, 'relics_obtained': ['HornCleat']}
{'damage_healed': 0.0, 'max_hp_gain': 0.0, 'max_hp_loss': 0.0, 'gold_gain': 0.0, 'event_name': 'Duplicator', 'player_choice': 'Copied', 'floor': 19.0, 'gold_loss': 0.0, 'damage_taken': 0.0, 'cards_obtained': ['Rage']}
{'damage_healed': 0.0, 'max_hp_gain': 0.0, 'max_hp_loss': 0.0, 'gold_gain': 0.0, 'cards_transformed': ['Strike_R'], 'event_name': 'Transmorgrifier', 'player_choice': 'Transformed', 'floor': 30.0, 'gold_loss': 0.0, 'damage_taken': 0.0, 'cards_obtained': ['Armaments']}
{'damage_healed': 0, 'gold_gain': 0, 'player_choice': 'Ignored', 'damage_taken': 0, 'max_hp_gain': 0, 'max_hp_loss': 0, 'event_name': 'Mysterious Sphere', 'floor': 38, 'gold_loss': 0}
{'damage_healed': 0, 'gold_gain': 0, 'player_choice': 'Embrace Madness', 'damage_tak

In [52]:
#Searching for all keys in the event_choices field across all Ironclad runs, to see what kind of data it contains

import collections

event_field_counts = collections.Counter()
total_event_entries = 0

for r in ironclad_runs:
    for e in r.get('event_choices', []):
        total_event_entries += 1
        event_field_counts.update(e.keys())

print("Total event_choices entries across all runs:", total_event_entries)
for key, count in sorted(event_field_counts.items(), key=lambda x: -x[1]):
    print(f"{key}: {count}/{total_event_entries} ({count/total_event_entries:.1%})")

Total event_choices entries across all runs: 1268
player_choice: 1268/1268 (100.0%)
damage_taken: 1268/1268 (100.0%)
event_name: 1268/1268 (100.0%)
floor: 1268/1268 (100.0%)
damage_healed: 1261/1268 (99.4%)
gold_gain: 1261/1268 (99.4%)
max_hp_gain: 1261/1268 (99.4%)
max_hp_loss: 1261/1268 (99.4%)
gold_loss: 1261/1268 (99.4%)
relics_obtained: 265/1268 (20.9%)
cards_obtained: 265/1268 (20.9%)
cards_removed: 209/1268 (16.5%)
cards_upgraded: 143/1268 (11.3%)
cards_transformed: 43/1268 (3.4%)
relics_lost: 11/1268 (0.9%)
potions_obtained: 4/1268 (0.3%)


In [53]:
# Checking if transformed cards constistently remove and add the same number of cards, and if the cards_transformed and cards_obtained fields are consistent

transform_examples = []
for r in ironclad_runs:
    for e in r.get('event_choices', []):
        if 'cards_transformed' in e:
            transform_examples.append(e)

print(len(transform_examples))
for e in transform_examples[:5]:
    print(e.get('cards_transformed'), '->', e.get('cards_obtained'))

43
['Fire Breathing'] -> ['Second Wind']
['Strike_R'] -> ['Armaments']
['Defend_R', 'Defend_R'] -> ['Rupture', 'Bloodletting']
['Strike_R', 'Strike_R'] -> ['Bludgeon', 'Blood for Blood']
['Strike_R'] -> ['Metallicize']


**Result:** `event_choices` indeed add/remove/transform cards, while documenting the changes of the deck in a consistent way.

### Question: Is the `cards_upgraded` field consistent (not changing the card count of the deck)?

In [54]:
# Checking if upgraded cards constistently remove and add the same number of cards, and if the cards_upgraded and cards_obtained fields are consistent

upgrade_examples = []
for r in ironclad_runs:
    for e in r.get('event_choices', []):
        if 'cards_upgraded' in e:
            upgrade_examples.append(e)

print(len(upgrade_examples))
for e in upgrade_examples[:8]:
    print(e.get('cards_upgraded'), "| obtained:", e.get('cards_obtained'), "| removed:", e.get('cards_removed'))

143
['Defend_R', 'Strike_R'] | obtained: None | removed: None
['Strike_R'] | obtained: None | removed: None
['Strike_R', 'Strike_R', 'Strike_R', 'Defend_R'] | obtained: None | removed: None
['Strike_R', 'Defend_R'] | obtained: None | removed: None
['Defend_R', 'Bash'] | obtained: None | removed: None
['Clash'] | obtained: None | removed: None
['Defend_R', 'Defend_R'] | obtained: None | removed: None
['Bash'] | obtained: None | removed: None


In [55]:
# INvestigating if there are any upgrade events that also have obtained or removed cards, which would be unexpected

co_occurrences = sum(1 for e in upgrade_examples if e.get('cards_obtained') or e.get('cards_removed'))
print(co_occurrences, "out of", len(upgrade_examples), "upgrade events also had obtained/removed")

2 out of 143 upgrade events also had obtained/removed


In [56]:
# Checking the two events that had both upgraded and obtained/removed cards, to see if there's any pattern

overlapping = [e for e in upgrade_examples if e.get('cards_obtained') or e.get('cards_removed')]
for e in overlapping:
    print(e)

{'cards_removed': ['Clumsy'], 'damage_healed': 0, 'gold_gain': 0, 'player_choice': 'Upgrade and Remove', 'damage_taken': 0, 'max_hp_gain': 0, 'max_hp_loss': 0, 'event_name': 'Designer', 'floor': 46, 'gold_loss': 90, 'cards_upgraded': ['Strike_R']}
{'cards_removed': ['Flex'], 'damage_healed': 0.0, 'max_hp_gain': 0.0, 'max_hp_loss': 0.0, 'gold_gain': 0.0, 'event_name': 'Designer', 'player_choice': 'Upgrade and Remove', 'floor': 20.0, 'gold_loss': 110.0, 'damage_taken': 0.0, 'cards_upgraded': ['Impervious']}


**Result:** `cards_upgraded` co-occurs with `cards_removed`/`cards_obtained` in 2/143 cases, both from the `'Designer'` event with `player_choice: 'Upgrade and Remove'`, a single event offering both an upgrade and a removal as one choice, applied to two different cards (no overlap/double-counting). Confirms `cards_upgraded` is safe to ignore for deck composition/count tracking in all cases. Co-occurring `cards_removed` is handled the same way as any other event-driven removal.

### Decision: The logs contain floor-indexed information for the major observed sources of mid-run deck modification. The exploratory reconstruction will attempt to reconstruct deck composition by replaying the following logged modifications in floor order:
- Starting deck (Neow-adjusted, per `neow_bonus`)
- `card_choices` picks (additions, excluding `SKIP`)
- `items_purged` (removals, validated: 100% occur on shop floors)
- `event_choices`: `cards_obtained` (additions), `cards_removed` (removals), `cards_transformed`↔`cards_obtained` pairs (validated 1-to-1/N-to-N across all 43 cases), `cards_upgraded` (in-place, ignored for composition tracking — validated independent of add/remove in 141/143 cases, with 2 known co-occurring-but-non-overlapping exceptions).

**Remaining limitation:** this reconstruction has been validated structurally (fields exist, align, and pair consistently) but not yet end-to-end against `master_deck` on real runs. Full validation will occur once the reconstruction function is implemented (Phase 1 build), comparing reconstructed final deck against actual `master_deck` across all runs to catch any remaining discrepancies.

## Summary of Findings
| Finding    | Status | Action    |
| :---        |    :----:   |          ---: | 
| F-001 | Confirmed | Use 'path_per_floor' | 
| F-002 | Confirmed | Use .get() for optional keys | 
| F-003 | Confirmed | Keep abandoned runs with flag | 
| F-004 | Confirmed | Reconstruct via `neow_bonus` + `card_choices` + `items_purged` + `event_choices` | 


# 4. Timeline reconstruction

# 5. Pseudocode

# 6. Exploratory implementation

# 7. Manual validation

# 8. Open questions